# 01 — Data Understanding
MarketMind AI — Milestone 2 — Customer Purchasing Behavior & Engagement Analysis

Inspects the raw Olist CSV files and documents schema, identifiers, and temporal range. No transformations happen in this notebook — pure inspection.

In [1]:
import pandas as pd
import os

RAW = "../data/raw"
files = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}
dfs = {name: pd.read_csv(os.path.join(RAW, fname)) for name, fname in files.items()}
{name: df.shape for name, df in dfs.items()}

{'customers': (99441, 5),
 'orders': (99441, 8),
 'order_items': (112650, 7),
 'order_payments': (103886, 5),
 'products': (32951, 9),
 'sellers': (3095, 4),
 'category_translation': (71, 2)}

In [2]:
for name, df in dfs.items():
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(df.dtypes.to_dict())
    print()

customers: 99,441 rows x 5 cols
{'customer_id': <StringDtype(storage='python', na_value=nan)>, 'customer_unique_id': <StringDtype(storage='python', na_value=nan)>, 'customer_zip_code_prefix': dtype('int64'), 'customer_city': <StringDtype(storage='python', na_value=nan)>, 'customer_state': <StringDtype(storage='python', na_value=nan)>}

orders: 99,441 rows x 8 cols
{'order_id': <StringDtype(storage='python', na_value=nan)>, 'customer_id': <StringDtype(storage='python', na_value=nan)>, 'order_status': <StringDtype(storage='python', na_value=nan)>, 'order_purchase_timestamp': <StringDtype(storage='python', na_value=nan)>, 'order_approved_at': <StringDtype(storage='python', na_value=nan)>, 'order_delivered_carrier_date': <StringDtype(storage='python', na_value=nan)>, 'order_delivered_customer_date': <StringDtype(storage='python', na_value=nan)>, 'order_estimated_delivery_date': <StringDtype(storage='python', na_value=nan)>}

order_items: 112,650 rows x 7 cols
{'order_id': <StringDtype(stor

## Customer identifier analysis

Two customer identifiers exist in this dataset. We determine which one is the correct grain for customer-level behavioral analysis.

In [3]:
cust = dfs["customers"]
print("customer_id unique:", cust['customer_id'].nunique())
print("customer_unique_id unique:", cust['customer_unique_id'].nunique())
print("rows per customer_unique_id (avg):", len(cust)/cust['customer_unique_id'].nunique())

customer_id unique: 99441
customer_unique_id unique: 96096
rows per customer_unique_id (avg): 1.0348089410589412


**Finding:** `customer_id` is a per-ORDER surrogate key — Olist issues a new one for every order to anonymize the public dataset. `customer_unique_id` is the true, persistent real-world customer identifier.

**Decision:** `customer_unique_id` is used as the grain for all customer-level behavioral analysis, RFM, and segmentation features throughout this project. `customer_id` is used only as the join key between customers and orders.

## Temporal range

In [4]:
orders = dfs["orders"].copy()
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
print("Min date:", orders["order_purchase_timestamp"].min())
print("Max date:", orders["order_purchase_timestamp"].max())
print("Invalid timestamps:", orders["order_purchase_timestamp"].isna().sum())

Min date: 2016-09-04 21:15:19
Max date: 2018-10-17 17:30:18
Invalid timestamps: 0


## order_status distribution

In [5]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Full schema mapping (missing values, duplicates, uniqueness per column) is documented in `../SCHEMA_MAPPING.md`, generated from this same inspection.